# G1b — cancer-variant depletion against a COSMIC mutational spectrum

**Pre-registered in `docs/preregistration/G1.md`, committed before this notebook was written.** Every threshold, every arm, and the reading of the result come from that file.

B6 reported that observed SH3 cancer variants are depleted of predicted fold-breaking substitutions — 5.93 % against 9.18 % expected — and said plainly what the test could not do: *mutational context was not modelled, and a trinucleotide-aware null is needed before claiming selection.* This is that test.

This notebook is one of a pair. It differs from its twin in **one cell** — the spectrum — and shares everything else as code, not as intention: `g1_core.py` is embedded below byte-for-byte and its SHA-256 is asserted, so any difference in the result is attributable to the spectrum rather than to an implementation slip.

| | |
|---|---|
| Spectrum | COSMIC SBS1 + SBS5, equal mixture |
| Independent of the test set? | yes |
| Predictions | B6's frozen virtual DMS, 22 Sep 2026. Never re-fitted |
| The call | `fold_lo >= 1`, exactly as B3 defined it |

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json, sys, hashlib
from pathlib import Path
import pandas as pd, numpy as np

DRIVE = Path('/content/drive/MyDrive')

def pick(*names, what=''):
    for n in names:
        if (DRIVE / n).exists(): return DRIVE / n
    raise FileNotFoundError(f'No {what} folder in MyDrive. Tried: ' + ', '.join(names))

PROJECT = pick('sh3_repro/extended', 'SH3_extended', 'SH3 extended', what='project')
FROZEN  = pick('sh3_repro', what='frozen-run')          # READ ONLY
FROZEN_IN = PROJECT / 'input' / 'frozen'
IN      = PROJECT / 'input' / 'external'
OUT     = PROJECT / 'results' / 'G1_mutational_context'
OUT.mkdir(parents=True, exist_ok=True)

def frozen(rel, bundled):
    p = FROZEN / 'results' / rel
    return p if p.exists() else FROZEN_IN / 'results' / bundled

print('project:', PROJECT); print('frozen :', FROZEN)

## 1 — The shared machinery

`tools/g1_core.py`, written to disk from a string this generator read at build time. The hash assertion is the point: without it, 'both notebooks run the same code' would be a claim rather than a check.

In [ ]:
# g1_core.py, embedded byte-for-byte by tools/nb_G1.py at generation time.
# Part 9.1: the two notebooks must run IDENTICAL machinery, so this is written
# from one source and its hash is asserted below rather than trusted.
core_src = r'''"""
G1 — shared machinery for both spectrum notebooks.

`docs/preregistration/G1.md` Part 9.1 requires that the eligible variant set, the
domain-specific opportunity counts, the B6 predictions, the draw procedure, the test
statistic and the seed be *shared code*, not a shared intention: two notebooks that
merely look alike would leave any difference in the result ambiguous between the
spectrum and an implementation slip.

So everything except the spectrum lives here.  `G1a_observed_spectrum` and
`G1b_cosmic_spectrum` differ in exactly one call:

    spectrum = g1_core.observed_spectrum(ctx)      # G1a
    spectrum = g1_core.cosmic_spectrum(ctx, path)  # G1b

Both notebooks embed this file byte-for-byte and assert its SHA-256, so "identical
machinery" is checkable rather than asserted.

What the arms are (Part 10):

    3a  uniform over amino-acid substitutions  -- B6's null; MUST reproduce 9.18 %
    3b  uniform over base changes              -- + reachability multiplicity
    1/2 spectrum-weighted over base changes    -- + trinucleotide context

Only 1/2 carries a verdict.  3a and 3b carry none; they exist so that any change in
the null rate can be attributed to a step rather than asserted.
"""
from __future__ import annotations

import gzip
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 20260928
N_DRAWS = 10_000
CALL = 'fold_lo'          # destabilising iff fold_lo >= 1, exactly as B3 defined it
CALL_THRESHOLD = 1.0

PURINES = {'A', 'G'}
COMP = str.maketrans('ACGTN', 'TGCAN')


# ─────────────────────────────────────────────────────────────────────────────
# 96-channel classification
# ─────────────────────────────────────────────────────────────────────────────

def channels() -> list[str]:
    """The 96 SBS channels in COSMIC's own row order."""
    out = []
    for ref in 'CT':
        for alt in 'ACGT':
            if alt == ref:
                continue
            for up in 'ACGT':
                for down in 'ACGT':
                    out.append(f'{up}[{ref}>{alt}]{down}')
    return sorted(out)


def to_channel(up: str, ref: str, alt: str, down: str) -> str | None:
    """Pyrimidine-referenced 96-channel label, or None if any base is not ACGT.

    A mutation and its reverse complement are the same event observed from the
    other strand, so channels are defined with a C or T in the middle.
    """
    if ref in PURINES:
        up, ref, alt, down = (down.translate(COMP), ref.translate(COMP),
                              alt.translate(COMP), up.translate(COMP))
    if ref not in 'CT' or alt not in 'ACGT' or alt == ref:
        return None
    if up not in 'ACGT' or down not in 'ACGT':
        return None
    return f'{up}[{ref}>{alt}]{down}'


# ─────────────────────────────────────────────────────────────────────────────
# Genome
# ─────────────────────────────────────────────────────────────────────────────

def read_flanks(fasta: Path, positions: dict[str, set[int]]) -> dict[tuple, str]:
    """Bases at every position in `positions`, by one streaming pass.

    The Ensembl primary assembly ships as plain gzip, not BGZF, so there is no
    random access and no index; one pass is the whole cost.  We ask for pos-1,
    pos and pos+1 together so the trinucleotide context comes out of the same pass.
    """
    want = {c: set() for c in positions}
    for c, ps in positions.items():
        for p in ps:
            want[c].update((p - 1, p, p + 1))

    got: dict[tuple, str] = {}
    cur, off, targets, maxpos = None, 0, None, 0
    with gzip.open(fasta, 'rt') as fh:
        for line in fh:
            if line[0] == '>':
                cur = line[1:].split()[0]
                off, targets = 0, want.get(cur)
                maxpos = max(targets) if targets else 0
                continue
            if not targets:
                continue
            if off >= maxpos:
                off += len(line) - 1
                continue
            s = line.rstrip('\n')
            lo = off + 1
            for p in range(lo, off + len(s) + 1):
                if p in targets:
                    got[(cur, p)] = s[p - lo].upper()
            off += len(s)
    return got


def add_context(df: pd.DataFrame, flanks: dict[tuple, str]) -> pd.DataFrame:
    """Attach `up`, `down` and `channel`; assert the reference base agrees.

    The reference check is the one number this reduction does not control, and X1
    is the reason it is demanded.
    """
    key = list(zip(df['chrom'], df['pos_grch38']))
    ref_seen = np.array([flanks.get((c, p), 'N') for c, p in key])
    bad = int((ref_seen != df['ref'].to_numpy()).sum())
    assert bad == 0, (
        f'{bad} rows whose GRCh38 base disagrees with the recorded ref allele; '
        'the coordinates are wrong and G1 must not proceed')

    up = np.array([flanks.get((c, p - 1), 'N') for c, p in key])
    down = np.array([flanks.get((c, p + 1), 'N') for c, p in key])
    out = df.copy()
    out['up'], out['down'] = up, down
    out['channel'] = [to_channel(u, r, a, d) for u, r, a, d
                      in zip(up, df['ref'], df['alt'], down)]
    return out


# ─────────────────────────────────────────────────────────────────────────────
# Pools
# ─────────────────────────────────────────────────────────────────────────────

def build_context(*, observed: pd.DataFrame, pool: pd.DataFrame,
                  b6: pd.DataFrame, fasta: Path) -> dict:
    """Everything both notebooks share.

    observed : atlas variants with one verified GRCh38 coordinate each
    pool     : every in-domain missense base change, from dbNSFP
    b6       : the frozen virtual DMS -- predictions, never re-fitted
    """
    b6 = b6.copy()
    call = (b6[CALL] >= CALL_THRESHOLD).astype(np.int8)
    if 'destabilising' in b6:
        # B6 ships its own column. Recomputing it from fold_lo and checking the two
        # agree costs nothing and catches a wrong file or a changed threshold.
        disagree = int((call != b6['destabilising'].astype(np.int8)).sum())
        assert disagree == 0, (
            f'{disagree} rows where fold_lo >= 1 disagrees with B6\'s own '
            "`destabilising` column; this is not the frozen call B3 defined")
    b6['destabilising'] = call
    keys = ['gene', 'pos_protein', 'wt_aa', 'mut_aa']
    b6k = b6.rename(columns={'pos': 'pos_protein'})[
        keys + ['domain_key', 'destabilising', CALL]].drop_duplicates(keys)

    n_obs_in, n_pool_in = len(observed), len(pool)
    observed = observed.merge(b6k, on=keys, how='inner', validate='m:1')
    pool = pool.merge(b6k, on=keys, how='inner', validate='m:1')

    pos = {}
    for d in (observed, pool):
        for c, p in zip(d['chrom'], d['pos_grch38']):
            pos.setdefault(c, set()).add(int(p))
    flanks = read_flanks(fasta, pos)
    observed, pool = add_context(observed, flanks), add_context(pool, flanks)

    # Part 10: a base change with no definable channel (N in the flank) cannot be
    # weighted by any spectrum, so it leaves both the pool and the observed set --
    # dropping it from one only would compare different populations.
    n_obs_ctx, n_pool_ctx = len(observed), len(pool)
    observed = observed[observed['channel'].notna()].reset_index(drop=True)
    pool = pool[pool['channel'].notna()].reset_index(drop=True)

    # 3a's unit: one item per amino-acid substitution, not per base change.
    sub = ['domain_key', 'pos_protein', 'wt_aa', 'mut_aa']
    subs = pool.drop_duplicates(sub).reset_index(drop=True)
    mult = pool.groupby(sub, sort=False).size()

    ctx = {
        'observed': observed, 'pool': pool, 'subs': subs,
        'counts': {
            'observed_in': n_obs_in, 'pool_in': n_pool_in,
            'observed_after_b6_join': n_obs_ctx, 'pool_after_b6_join': n_pool_ctx,
            'observed_eligible': len(observed), 'pool_base_changes': len(pool),
            'pool_substitutions': len(subs),
            'domains_observed': observed['domain_key'].nunique(),
            'domains_pool': pool['domain_key'].nunique(),
        },
        'multiplicity': {int(k): int(v) for k, v in mult.value_counts().items()},
        'opportunity': pool['channel'].value_counts().reindex(channels()).fillna(0)
                           .astype(int).to_dict(),
    }
    return ctx


# ─────────────────────────────────────────────────────────────────────────────
# Spectra -- the only thing the two notebooks do differently
# ─────────────────────────────────────────────────────────────────────────────

def _normalise(s: pd.Series) -> pd.Series:
    s = s.reindex(channels()).fillna(0.0).astype(float)
    assert s.sum() > 0, 'spectrum is empty'
    return s / s.sum()


def observed_spectrum(ctx: dict) -> pd.Series:
    """G1a: the 96-channel distribution of the variants under study.

    Part 6 and 9.4: this is estimated from the same variants whose damage is being
    tested, so it is NOT independent of the test set.  It is here because it
    reproduces whatever mixture of processes the atlas aggregates without having to
    estimate it -- and G1b exists because of the dependence, not in spite of it.
    """
    return _normalise(ctx['observed']['channel'].value_counts())


def cosmic_spectrum(path: Path, sigs=('SBS1', 'SBS5')) -> pd.Series:
    """G1b: an equal mixture of clock-like COSMIC signatures, fixed externally.

    Independent of the test set, at the price of assuming clock-like processes
    dominate across the tumour types the atlas aggregates.  Neither spectrum is
    automatically the safer choice; Part 9.4 says how they are compared.
    """
    t = pd.read_csv(path, sep='\t').set_index('Type')
    missing = [s for s in sigs if s not in t.columns]
    assert not missing, f'{missing} absent from the COSMIC table'
    return _normalise(t[list(sigs)].mean(axis=1))


def cosine(a: pd.Series, b: pd.Series) -> float:
    """Part 9.4's fit measure: how close an assumed spectrum is to the observed one."""
    x, y = _normalise(a).to_numpy(), _normalise(b).to_numpy()
    return float(x @ y / (np.linalg.norm(x) * np.linalg.norm(y)))


# ─────────────────────────────────────────────────────────────────────────────
# The draw
# ─────────────────────────────────────────────────────────────────────────────

def _draw(pool: pd.DataFrame, need: dict[str, int], weight: np.ndarray | None,
          rng: np.random.Generator) -> np.ndarray:
    """Destabilising count per draw, matched to the observed per-domain counts.

    Sampling is WITH replacement.  Each domain's pool (hundreds of candidates) is
    far larger than its observed count (tens), so the difference from B6's draw is
    negligible -- and 3a is the arm that has to prove that, by reproducing 9.18 %.
    """
    total = np.zeros(N_DRAWS, dtype=np.int32)
    dest = pool['destabilising'].to_numpy()
    for dom, idx in pool.groupby('domain_key', sort=False).indices.items():
        n = need.get(dom, 0)
        if n == 0:
            continue
        p = None
        if weight is not None:
            w = weight[idx]
            if w.sum() <= 0:
                continue          # no candidate in this domain carries any weight
            p = w / w.sum()
        picked = rng.choice(idx, size=(N_DRAWS, n), replace=True, p=p)
        total += dest[picked].sum(axis=1, dtype=np.int32)
    return total


def run_arm(ctx: dict, *, name: str, unit: str, spectrum: pd.Series | None) -> dict:
    """One arm.  `unit` is 'base_change' or 'substitution' (Part 10)."""
    obs = ctx['observed']
    pool = ctx['pool'] if unit == 'base_change' else ctx['subs']
    need = obs['domain_key'].value_counts().to_dict()

    weight = None
    if spectrum is not None:
        assert unit == 'base_change', 'a spectrum can only weight base changes'
        weight = spectrum.reindex(pool['channel']).to_numpy(dtype=float)

    rng = np.random.default_rng(SEED)
    counts = _draw(pool, need, weight, rng)
    n = len(obs)
    null = counts / n
    observed_rate = float(obs['destabilising'].mean())

    # one-sided, observed BELOW null -- the direction was declared in Part 6
    p = float((np.sum(counts <= obs['destabilising'].sum()) + 1) / (N_DRAWS + 1))
    return {
        'arm': name, 'unit': unit, 'n_observed': n, 'pool_size': len(pool),
        'observed_rate': observed_rate,
        'null_median': float(np.median(null)),
        'null_mean': float(null.mean()), 'null_sd': float(null.std(ddof=1)),
        'null_ci95': [float(np.percentile(null, 2.5)),
                      float(np.percentile(null, 97.5))],
        'gap_pp': float((np.median(null) - observed_rate) * 100),
        'p_one_sided_depleted': p,
        'draws': N_DRAWS, 'seed': SEED,
    }


def verdict(arm: dict, alpha: float) -> str:
    """Part 4's three outcomes.  The middle one is the one most exposed to hindsight,
    so it is named rather than left to prose."""
    if arm['p_one_sided_depleted'] >= alpha:
        return 'vanishes'
    return 'survives' if arm['gap_pp'] >= 0.5 * 3.25 else 'shrinks_but_survives'


def self_sha256() -> str:
    return hashlib.sha256(Path(__file__).read_bytes()).hexdigest()


def write_metrics(path: Path, payload: dict) -> None:
    payload = dict(payload, g1_core_sha256=self_sha256(), seed=SEED, draws=N_DRAWS)
    Path(path).write_text(json.dumps(payload, indent=2))
'''
Path('g1_core.py').write_text(core_src)

EXPECTED_SHA = '0846c3a66bba56e03cd27144f927a42a8f90eaabcc421e66ef745e6dca88c4f9'
got = hashlib.sha256(Path('g1_core.py').read_bytes()).hexdigest()
assert got == EXPECTED_SHA, (
    f'g1_core.py hashes to {got}, expected {EXPECTED_SHA}. The two notebooks would '
    'not be running the same code, so no comparison between them would be valid.')

sys.path.insert(0, '.')
import g1_core as G
print('g1_core sha256:', got[:16], '...  seed', G.SEED, ' draws', G.N_DRAWS)

## 2 — Which threshold applies

Part 9.3 lets the choice of primary spectrum be deferred, and charges for it in advance. The notebook **reads the pre-registration** and takes α = 0.025 if a line `PRIMARY SPECTRUM: …` is there, or α = 0.0125 for both notebooks if it is not. Choosing after seeing a p-value is therefore not possible — by then the file is already committed.

Which role *this* notebook plays needs its identity, and the identity is set in the spectrum cell — the single cell that differs between G1a and G1b. Naming it here as well would make two cells differ, and the whole comparison rests on exactly one differing. So the threshold is read here and the role is resolved in §8.

In [ ]:
# Part 9.3 -- the threshold is read from the pre-registration, not chosen here.
# If G1.md names a primary spectrum before the run, that arm takes alpha = 0.025
# and the other is a sensitivity analysis. If it names none, both are co-primary
# at 0.0125 (Bonferroni, family-wise 0.025). Deciding after seeing a p-value is
# therefore impossible: by then the file is already committed.
prereg = (PROJECT / 'docs' / 'preregistration' / 'G1.md')
text = prereg.read_text() if prereg.exists() else ''
MARK = 'PRIMARY SPECTRUM:'
named = [l.split(MARK, 1)[1].strip() for l in text.splitlines() if MARK in l]
PRIMARY = named[0] if named else None
ALPHA = 0.025 if PRIMARY else 0.0125
print(f'primary named in G1.md : {PRIMARY!r}')
print(f'alpha                  : {ALPHA}')

# Which role THIS notebook plays needs its identity, and the identity lives in
# the spectrum cell -- the one cell that differs between G1a and G1b. Naming it
# here too would make two cells differ, and the comparison rests on exactly one
# differing. So IS_PRIMARY is resolved in section 8, once the spectrum is known.
print('role resolved in section 8, once the spectrum cell has named this notebook')

## 3 — The pinned inputs

Three files, none of them re-fitted here: B6's virtual DMS, the verified coordinate table from `tools/g1_coordinates.py`, and X1's dbNSFP subset, which is the only source of a genomic coordinate for every *possible* in-domain missense SNV — that is, for the null pool.

In [ ]:
# ---- the three pinned inputs (Part 2). Nothing here is re-fitted. ----
b6 = pd.read_parquet(frozen('B6_virtual_saturation_mutagenesis/sh3_virtual_dms.parquet',
                            'sh3_virtual_dms.parquet'))
print('B6 virtual DMS:', len(b6), 'rows')
assert len(b6) == 366168, f'B6 has {len(b6)} rows, expected 366,168 -- wrong file'

obs = pd.read_csv(OUT / 'atlas_grch38_coordinates.tsv', sep='\t')
obs['pos_protein'] = obs['pos_protein'].astype(int)
print('observed with a verified coordinate:', len(obs))

# The null pool: every in-domain missense base change dbNSFP carries.
pool_p = PROJECT / 'results' / 'X1_dbnsfp_subset' / 'dbnsfp_sh3_domains.parquet'
assert pool_p.exists(), (
    f'{pool_p} not found. X1 must be on Drive: it is the only source of a genomic '
    'coordinate for every possible in-domain missense SNV, which is the null pool.')
pool = pd.read_parquet(pool_p)
print('dbNSFP in-domain rows:', len(pool))

## 4 — Building the null pool

Part 10: the null draws **base changes**, not amino-acid substitutions. One amino-acid change can be reached by several base changes, each with its own trinucleotide context, so a context-weighted null is meaningless at the amino-acid level.

The pool is aligned on the atlas's own UniProt accession, the same way the observed set was. Taking element `[0]` of dbNSFP's per-transcript fields is the defect that cost X1 a run.

In [ ]:
# dbNSFP's per-transcript fields are semicolon-separated and index-aligned.
# Taking element [0] is the defect that cost X1 a run, so the pool is built the
# same way tools/g1_coordinates.py built the observed set: aligned on the UniProt
# accession the atlas numbered its residues against.
dom = pd.read_csv(FROZEN_IN / 'results' / 'SH3_cancer_master.tsv', sep='\t',
                  usecols=['gene', 'uniprot']).drop_duplicates()
acc = dict(zip(dom['gene'], dom['uniprot'].str.split('-').str[0]))

rows = []
for c, p, rf, al, ar, aa, ap, gn, up in zip(
        pool['#chr'], pool['pos(1-based)'], pool['ref'], pool['alt'],
        pool['aaref'], pool['aaalt'], pool['aapos'], pool['genename'],
        pool['Uniprot_acc']):
    if ar == '.' or aa == '.' or ar == aa:
        continue
    ps, us = str(ap).split(';'), str(up).split(';')
    for g in {x for x in str(gn).split(';') if x and x != '.'}:
        want = acc.get(g)
        if want is None:
            continue
        for j, q in enumerate(ps):
            if not q or q == '.' or j >= len(us):
                continue
            if us[j].split('-')[0] != want:
                continue
            rows.append((g, int(q), ar, aa, str(c), int(p), rf, al))
            break

pool = pd.DataFrame(rows, columns=['gene', 'pos_protein', 'wt_aa', 'mut_aa',
                                   'chrom', 'pos_grch38', 'ref', 'alt'])
pool = pool.drop_duplicates()
print('null pool, UniProt-aligned base changes:', len(pool))

## 5 — Trinucleotide context

One streaming pass over GRCh38 for both pools at once. `add_context` asserts that the reference base agrees at every position — the one number this reduction does not control.

In [ ]:
# One streaming pass over GRCh38 gives every trinucleotide context. The assembly
# ships as plain gzip, not BGZF, so there is no index and no region query.
FASTA = IN / 'genome' / 'Homo_sapiens.GRCh38.dna.primary_assembly.fa.gz'
GENOME_BYTES = 881964081          # DATA_PROVENANCE.md §7
GENOME_URL = ('https://ftp.ensembl.org/pub/current_fasta/homo_sapiens/dna/'
              'Homo_sapiens.GRCh38.dna.primary_assembly.fa.gz')
if not FASTA.exists():
    # 842 MB. Colab pulls it from Ensembl far faster than a home connection can
    # push it to Drive, so this is the preferred route rather than a fallback.
    local = Path('/content/GRCh38.fa.gz')
    if not (local.exists() and local.stat().st_size == GENOME_BYTES):
        print('fetching GRCh38 from Ensembl (~842 MB)...')
        !wget -q --show-progress -O {local} {GENOME_URL}
    FASTA = local
assert FASTA.stat().st_size == GENOME_BYTES, (
    f'{FASTA} is {FASTA.stat().st_size} bytes, expected {GENOME_BYTES}. Ensembl moved to a new release, or the download was truncated -- either way the contexts would not be the ones DATA_PROVENANCE §7 pins.')
print('genome:', FASTA)

ctx = G.build_context(observed=obs, pool=pool, b6=b6, fasta=FASTA)
print(json.dumps(ctx['counts'], indent=2))
print('\nbase changes reaching one amino-acid substitution:')
print(json.dumps(ctx['multiplicity'], indent=2))

assert ctx['counts']['observed_eligible'] >= 3000, (
    'fewer than 3,000 eligible observed variants -- Part 4 says numbers are shown and NO verdict is given')

## 6 — The controls, which decide whether anything below is interpretable

**3a must reproduce B6's 9.18 %.** If it does not, this implementation differs from B6 in something that is not the trinucleotide weighting, and the spectrum arm means nothing. The notebook stops.

**3b is not an error if it differs from 3a.** That difference is reachability multiplicity — how many distinct base changes reach each amino-acid substitution — isolated from sequence context, and it is reported as its own number.

```
3a  uniform over substitutions   -- B6's null
3b  uniform over base changes    -- + reachability multiplicity
1/2 spectrum-weighted            -- + trinucleotide context
```

In [ ]:
# ---- Controls (Part 10). Neither carries a verdict. ----
a3a = G.run_arm(ctx, name='3a uniform over substitutions', unit='substitution',
                spectrum=None)
a3b = G.run_arm(ctx, name='3b uniform over base changes', unit='base_change',
                spectrum=None)
for a in (a3a, a3b):
    print(f"{a['arm']:34s} pool {a['pool_size']:7d}  null median "
          f"{a['null_median']:.4f}  observed {a['observed_rate']:.4f}")

# Part 3: this is the stop condition, and it attaches to 3a alone.
B6_NULL = 0.09177
off = abs(a3a['null_median'] - B6_NULL)
print(f'\n3a vs B6 published null: {a3a["null_median"]:.5f} vs {B6_NULL:.5f} (off by {off:.5f})')
assert off <= 0.005, (
    f'3a gives {a3a["null_median"]:.5f} where B6 published {B6_NULL:.5f}. This implementation differs from B6 in something that is NOT the trinucleotide weighting, so the spectrum arms are not interpretable. Stop and find it.')
print('control passes -- the spectrum arms mean what they claim')

print(f'\nreachability multiplicity alone moves the null by {(a3b["null_median"] - a3a["null_median"])*100:+.2f} pp')

## 7 — The spectrum

The one cell that differs between the two notebooks.

In [ ]:
# ======== THE ONLY CELL THAT DIFFERS BETWEEN G1a AND G1b ========
# G1b -- an equal mixture of the clock-like COSMIC signatures SBS1 and SBS5,
# fixed externally and estimated from nothing in this project.
#
# Independent of the test set, at the price of assuming clock-like processes
# dominate across the tumour types the atlas aggregates. That is a real
# assumption and it is why this arm is not automatically the safer one.
NOTEBOOK = 'G1b'
NOTEBOOK_SPECTRUM = 'cosmic'
ARM_NAME = '2 COSMIC SBS1+SBS5'
SIGS = IN / 'signatures' / 'COSMIC_v3.6_SBS_GRCh38.txt'
assert SIGS.exists(), f'{SIGS} not found'
spectrum = G.cosmic_spectrum(SIGS, sigs=('SBS1', 'SBS5'))
print('COSMIC SBS1+SBS5, top channels:')
print(spectrum.sort_values(ascending=False).head(10).to_string())

## 8 — The test

One-sided, observed **below** null, the direction declared in Part 6. The verdict label comes from `g1_core.verdict()` so that Part 11's 1.63 pp boundary between *survives* and *shrinks but survives* is applied by code rather than by hand.

In [ ]:
# Now that the spectrum cell has named this notebook, section 2's threshold can
# be turned into a role. Still nothing chosen here: PRIMARY came from the
# committed pre-registration, not from anything computed below.
IS_PRIMARY = (PRIMARY is None) or PRIMARY.lower().startswith(NOTEBOOK_SPECTRUM)
print(f'this notebook : {NOTEBOOK_SPECTRUM}')
print('role          : '
      + ('co-primary (no primary named)' if PRIMARY is None else
         'PRIMARY' if IS_PRIMARY else 'sensitivity analysis -- no verdict'))

# ---- The arm that carries the verdict ----
main = G.run_arm(ctx, name=ARM_NAME, unit='base_change', spectrum=spectrum)
obs_spec = G.observed_spectrum(ctx)
fit = G.cosine(spectrum, obs_spec)

print(json.dumps(main, indent=2))
print(f'\ncosine fit to the observed 96-channel distribution: {fit:.4f}')
print(f'context alone moves the null by {(main["null_median"] - a3b["null_median"])*100:+.2f} pp')

v = G.verdict(main, ALPHA)
print(f'\nalpha {ALPHA}   p {main["p_one_sided_depleted"]:.5f}   gap {main["gap_pp"]:+.2f} pp   ->  {v.upper()}')
if not IS_PRIMARY:
    print('This notebook is the SENSITIVITY analysis -- reported, no verdict.')

## 9 — Who was excluded

6.2 % of the atlas has no unambiguous coordinate, and the exclusion plausibly correlates with genes carrying many transcripts. Part 6 requires this comparison to be **computed and reported whether or not it changes anything** — it is not a condition on the verdict, because the rule for that would have to be invented after seeing it.

In [ ]:
# Part 6 requires this whether or not it changes anything, so that the choice to
# report it is not made after seeing it.
atlas = pd.read_csv(FROZEN_IN / 'results' / 'SH3_cancer_master.tsv', sep='\t')
k = ['gene', 'pos_protein', 'wt_aa', 'mut_aa']
b6k = b6.rename(columns={'pos': 'pos_protein'}).copy()
b6k['destabilising'] = (b6k[G.CALL] >= G.CALL_THRESHOLD).astype(int)
full = atlas[k].merge(b6k[k + ['destabilising']], on=k, how='inner')
kept = set(map(tuple, ctx['observed'][k].values))
mask = [tuple(r) in kept for r in full[k].values]
inc, exc = full[mask], full[[not m for m in mask]]
print(f'included {len(inc):5d}  destabilising {inc["destabilising"].mean():.4f}')
print(f'excluded {len(exc):5d}  destabilising {exc["destabilising"].mean():.4f}' if len(exc) else 'excluded     0')
excl = {'n_included': int(len(inc)), 'n_excluded': int(len(exc)),
        'rate_included': float(inc['destabilising'].mean()),
        'rate_excluded': float(exc['destabilising'].mean()) if len(exc) else None}

## 10 — Write the machine-written record

Part 7: the spectrum is published as normalised proportions, never raw counts; the per-variant tables stay on the laptop. Nothing in `RESULTS_*.md` is typed by hand; the interpretation goes in `ANALYSIS_G1.md` once both notebooks and `tools/g1_compare.py` have run.

In [ ]:
payload = {'notebook': NOTEBOOK, 'spectrum': NOTEBOOK_SPECTRUM,
           'primary_named_in_prereg': PRIMARY, 'is_primary': bool(IS_PRIMARY),
           'alpha': ALPHA, 'verdict': v,
           'main': main, 'control_3a': a3a, 'control_3b': a3b,
           'b6_published_null': B6_NULL,
           'cosine_fit_to_observed': fit,
           'counts': ctx['counts'], 'multiplicity': ctx['multiplicity'],
           'exclusion_check': excl,
           'spectrum_used': {k: float(x) for k, x in spectrum.items()},
           'observed_spectrum': {k: float(x) for k, x in obs_spec.items()},
           'opportunity': ctx['opportunity']}
G.write_metrics(OUT / f'metrics_{NOTEBOOK}.json', payload)
print('wrote', OUT / f'metrics_{NOTEBOOK}.json')

# Part 7: the spectrum is published as normalised proportions, never raw counts,
# and the per-variant tables stay on the laptop.
lines = [f'# {NOTEBOOK} — machine-written results', '',
         f'Spectrum: **{NOTEBOOK_SPECTRUM}**. Role: **{"primary" if IS_PRIMARY else "sensitivity"}**. alpha = {ALPHA}.', '',
         '| arm | unit | pool | null median | observed | gap (pp) | p |',
         '|---|---|---|---|---|---|---|']
for a in (a3a, a3b, main):
    lines.append(f"| {a['arm']} | {a['unit']} | {a['pool_size']} | "
                 f"{a['null_median']:.4f} | {a['observed_rate']:.4f} | "
                 f"{a['gap_pp']:+.2f} | {a['p_one_sided_depleted']:.5f} |")
lines += ['', f'**Verdict ({NOTEBOOK_SPECTRUM}): {v}**', '',
          f'Cosine fit to the observed 96-channel distribution: {fit:.4f}.', '',
          f'g1_core sha256 `{G.self_sha256()[:16]}...`, seed {G.SEED}, {G.N_DRAWS} draws.']
(OUT / f'RESULTS_{NOTEBOOK}.md').write_text('\n'.join(lines))
print('wrote', OUT / f'RESULTS_{NOTEBOOK}.md')